In [3]:
import pandas as pd
import os
import random
from pyecharts import options as opts
from pyecharts.charts import Graph

# ================= ⚡ 严格配置区 =================
INPUT_FILE = 'ppmi_knowledge_triplets.csv'
OUTPUT_SUBSET = "ppmi_kg_subset_10.html"
OUTPUT_FULL = "ppmi_kg_full.html"

# 🎨 颜色策略：冷暖对立，区分内外
# [内部数据 - PPMI] -> 暖色系 (活跃、核心)
COLOR_INTERNAL_PATIENT = "#c23531"  # 鲜红 (病人)
COLOR_INTERNAL_DEMO    = "#d48265"  # 砖红 (年龄/性别)

# [语义桥梁] -> 中性色 (连接点)
COLOR_BRIDGE_PHENO     = "#2f4554"  # 深灰蓝 (症状 - 既是病人表现，也是医学概念)

# [外部知识 - PrimeKG] -> 冷色系 (客观、背景)
COLOR_EXTERNAL_GENE    = "#61a0a8"  # 青色 (基因)
COLOR_EXTERNAL_BIO     = "#749f83"  # 绿色 (生物过程)
COLOR_EXTERNAL_DISEASE = "#91c7ae"  # 浅绿 (疾病)
COLOR_EXTERNAL_PATHWAY = "#546570"  # 灰青 (通路)

# 建立严格的类型映射
TYPE_CONFIG = {
    # 内部
    'patient':            {'color': COLOR_INTERNAL_PATIENT, 'group': 'Internal (PPMI)'},
    'demographic':        {'color': COLOR_INTERNAL_DEMO,    'group': 'Internal (PPMI)'},
    # 桥梁
    'effect/phenotype':   {'color': COLOR_BRIDGE_PHENO,     'group': 'Bridge (Symptom)'},
    # 外部
    'gene/protein':       {'color': COLOR_EXTERNAL_GENE,    'group': 'External (PrimeKG)'},
    'biological_process': {'color': COLOR_EXTERNAL_BIO,     'group': 'External (PrimeKG)'},
    'disease':            {'color': COLOR_EXTERNAL_DISEASE, 'group': 'External (PrimeKG)'},
    'pathway':            {'color': COLOR_EXTERNAL_PATHWAY, 'group': 'External (PrimeKG)'},
    # 其他
    'other':              {'color': "#ccc",                 'group': 'Other'}
}

# ================= 1. 加载与严格分区统计 =================
def load_and_strict_report(filepath):
    print(f"🚀 [Step 1] 加载并分析图谱: {filepath} ...")
    if not os.path.exists(filepath):
        print("❌ 文件未找到！")
        return None

    df = pd.read_csv(filepath)
    
    # 1. 节点归类
    nodes_info = {} # {name: type}
    for _, row in df.iterrows():
        nodes_info[row['x_name']] = row['x_type']
        nodes_info[row['y_name']] = row['y_type']
        
    # 2. 边归类 (区分是 "诊断记录" 还是 "医学原理")
    # 诊断边: 包含 patient 的边
    # 知识边: 不包含 patient 的边
    df['edge_source'] = df.apply(lambda r: 'PPMI Record' if r['x_type']=='patient' else 'Bio Knowledge', axis=1)
    
    # --- 打印分区统计报告 ---
    print("\n" + "="*60)
    print("📊 知识图谱分区统计 (Knowledge Source Partition)")
    print("="*60)
    
    # 统计各组数量
    group_counts = {'Internal (PPMI)': 0, 'Bridge (Symptom)': 0, 'External (PrimeKG)': 0}
    detail_counts = {}
    
    for node, ntype in nodes_info.items():
        config = TYPE_CONFIG.get(ntype, TYPE_CONFIG['other'])
        group = config['group']
        
        # 累加组总数
        group_counts[group] = group_counts.get(group, 0) + 1
        
        # 累加详细类型数
        detail_key = f"{group} - {ntype}"
        detail_counts[detail_key] = detail_counts.get(detail_key, 0) + 1
        
    # 打印 - 内部数据
    print(f"🔴 [PPMI 内部数据空间] (共 {group_counts.get('Internal (PPMI)', 0)} 节点)")
    print(f"   说明: 这是你收集的私有临床数据")
    print(f"   - 病人 (Patient)       : {detail_counts.get('Internal (PPMI) - patient', 0)}")
    print(f"   - 人口学 (Demographic) : {detail_counts.get('Internal (PPMI) - demographic', 0)}")
    
    print("-" * 60)
    print(f"⚫ [语义桥梁 / 症状表型] (共 {group_counts.get('Bridge (Symptom)', 0)} 节点)")
    print(f"   说明: 连接 PPMI 与 PrimeKG 的锚点")
    print(f"   - 临床表型 (Phenotype) : {detail_counts.get('Bridge (Symptom) - effect/phenotype', 0)}")
    
    print("-" * 60)
    print(f"🔵 [PrimeKG 外部知识空间] (共 {group_counts.get('External (PrimeKG)', 0)} 节点)")
    print(f"   说明: 解释症状机制的通用医学知识")
    print(f"   - 基因/蛋白 (Gene)     : {detail_counts.get('External (PrimeKG) - gene/protein', 0)}")
    print(f"   - 生物过程 (BioProcess): {detail_counts.get('External (PrimeKG) - biological_process', 0)}")
    print(f"   - 疾病标签 (Disease)   : {detail_counts.get('External (PrimeKG) - disease', 0)}")
    print(f"   - 信号通路 (Pathway)   : {detail_counts.get('External (PrimeKG) - pathway', 0)}")
    
    print("-" * 60)
    print(f"🔗 [连接关系统计]")
    print(f"   - 诊断记录 (PPMI Records)   : {len(df[df['edge_source']=='PPMI Record'])} 条 (病人->症状/特征)")
    print(f"   - 医学原理 (Bio Mechanisms) : {len(df[df['edge_source']=='Bio Knowledge'])} 条 (症状->基因->机理)")
    print("="*60 + "\n")
    
    return df, nodes_info

# ================= 2. 可视化渲染 (按组着色) =================
def render_graph(df_viz, nodes_info, filename, title):
    print(f"🎨 正在渲染: {title} ...")
    
    nodes_set = set(df_viz['x_name']) | set(df_viz['y_name'])
    
    # 构建图例类别 (按组名显示，而不是按类型名，这样更清晰)
    # 我们希望图例显示: Patient(Internal), Gene(External) 等
    categories = []
    cat_name_map = {} # 原始类型 -> 图例名
    
    # 定义图例顺序
    legend_order = ['patient', 'demographic', 'effect/phenotype', 'gene/protein', 'biological_process', 'disease', 'pathway']
    
    for t in legend_order:
        if t in TYPE_CONFIG:
            cfg = TYPE_CONFIG[t]
            # 图例名称带上来源标记，区分更明显
            legend_name = f"{t.upper()} [{cfg['group'].split(' ')[0]}]"
            categories.append({"name": legend_name, "itemStyle": {"color": cfg['color']}})
            cat_name_map[t] = legend_name
            
    # 构建 ECharts 节点
    echarts_nodes = []
    cat_names_list = [c['name'] for c in categories]
    
    for node in nodes_set:
        ntype = nodes_info.get(node, 'other')
        legend_name = cat_name_map.get(ntype, f"OTHER")
        
        # 容错：如果该类型不在我们的预设列表中，归为 other
        if legend_name not in cat_names_list:
            # 简单处理，或者跳过
            cat_idx = 0 
        else:
            cat_idx = cat_names_list.index(legend_name)
            
        # 大小
        size = 10
        if ntype == 'patient': size = 30
        elif ntype == 'effect/phenotype': size = 18
        elif ntype == 'gene/protein': size = 15
        
        echarts_nodes.append({
            "name": node,
            "symbolSize": size,
            "category": cat_idx,
            "draggable": True,
            "value": ntype,
            # 只显示核心节点的标签
            "label": {"show": ntype in ['patient', 'disease', 'demographic']} 
        })
        
    # 构建边
    echarts_links = []
    for _, row in df_viz.iterrows():
        echarts_links.append({
            "source": row['x_name'],
            "target": row['y_name'],
            "value": row['relation']
        })
        
    c = (
        Graph(init_opts=opts.InitOpts(width="100%", height="950px", page_title=title))
        .add(
            "",
            echarts_nodes,
            echarts_links,
            categories=categories,
            layout="force",
            repulsion=1000,
            gravity=0.08,
            edge_symbol=[None, None],
            edge_label=opts.LabelOpts(is_show=False),
            is_roam=True,
            is_focusnode=True,
            tooltip_opts=opts.TooltipOpts(formatter="{b} <br/> {c}")
        )
        .set_global_opts(
            title_opts=opts.TitleOpts(title=title, subtitle="红色系=PPMI内部数据 | 蓝色系=外部医学知识"),
            legend_opts=opts.LegendOpts(orient="vertical", pos_left="2%", pos_top="5%")
        )
    )
    c.render(filename)
    print(f"✅ 文件已生成: {filename}")

# ================= 3. 执行 =================
def main():
    data = load_and_strict_report(INPUT_FILE)
    if data is None: return
    df, nodes_info = data
    
    # 1. 生成 10人 样本图
    all_pats = [n for n, t in nodes_info.items() if t == 'patient']
    if all_pats:
        pats = random.sample(all_pats, min(10, len(all_pats)))
        # 抽取逻辑
        df_l1 = df[df['x_name'].isin(pats) | df['y_name'].isin(pats)]
        neighbors = set(df_l1['x_name']) | set(df_l1['y_name'])
        # 仅向外延伸 (不含回到其他病人的边)
        df_l2 = df[
            (df['x_name'].isin(neighbors) | df['y_name'].isin(neighbors)) & 
            (df['x_type'] != 'patient') & (df['y_type'] != 'patient')
        ]
        df_sub = pd.concat([df_l1, df_l2]).drop_duplicates()
        render_graph(df_sub, nodes_info, OUTPUT_SUBSET, "PPMI 样本图谱 (10人)")
    
    # 2. 生成全量图
    render_graph(df, nodes_info, OUTPUT_FULL, "PPMI 完整多模态图谱")
    print("\n🎉 完成！请查看 HTML 文件，图例中已明确标注 [Internal] 和 [External]。")

if __name__ == "__main__":
    main()

🚀 [Step 1] 加载并分析图谱: ppmi_knowledge_triplets.csv ...

📊 知识图谱分区统计 (Knowledge Source Partition)
🔴 [PPMI 内部数据空间] (共 347 节点)
   说明: 这是你收集的私有临床数据
   - 病人 (Patient)       : 339
   - 人口学 (Demographic) : 8
------------------------------------------------------------
⚫ [语义桥梁 / 症状表型] (共 140 节点)
   说明: 连接 PPMI 与 PrimeKG 的锚点
   - 临床表型 (Phenotype) : 140
------------------------------------------------------------
🔵 [PrimeKG 外部知识空间] (共 320 节点)
   说明: 解释症状机制的通用医学知识
   - 基因/蛋白 (Gene)     : 210
   - 生物过程 (BioProcess): 108
   - 疾病标签 (Disease)   : 2
   - 信号通路 (Pathway)   : 0
------------------------------------------------------------
🔗 [连接关系统计]
   - 诊断记录 (PPMI Records)   : 1684 条 (病人->症状/特征)
   - 医学原理 (Bio Mechanisms) : 1464 条 (症状->基因->机理)

🎨 正在渲染: PPMI 样本图谱 (10人) ...
✅ 文件已生成: ppmi_kg_subset_10.html
🎨 正在渲染: PPMI 完整多模态图谱 ...
✅ 文件已生成: ppmi_kg_full.html

🎉 完成！请查看 HTML 文件，图例中已明确标注 [Internal] 和 [External]。
